In [ ]:
%pip install -qU pymongo python-dotenv

In [ ]:
from dotenv import load_dotenv
from pymongo import MongoClient
import os

load_dotenv()

client = MongoClient(os.getenv("MONGODB_URI"))

db = client["blogdb"]

collection = db["ai_news"]

# Identify duplicates


In [ ]:
pipeline = [
    {"$group": {"_id": "$title", "count": {"$sum": 1}}},
    {"$match": {"count": {"$gt": 1}}},
    {"$count": "duplicate_titles"},
]

result = collection.aggregate(pipeline, allowDiskUse=True)
duplicate_count = next(result, {}).get("duplicate_titles", 0)

print(f"Number of titles with duplicates: {duplicate_count}")

Number of titles with duplicates: 2344


In [ ]:
pipeline = [
    {
        "$group": {
            "_id": "$title",
            "count": {"$sum": 1},
            "samples": {"$push": {"url": "$url", "date": "$date"}},
        }
    },
    {"$match": {"count": {"$gt": 1}}},
    {"$sort": {"count": -1}},
    {"$limit": 20},  # Adjust this number to see more or fewer results
    {
        "$project": {
            "title": "$_id",
            "count": 1,
            "samples": {"$slice": ["$samples", 3]},  # Show up to 3 sample documents
        }
    },
]

results = collection.aggregate(pipeline, allowDiskUse=True)

for result in results:
    print(f"\nTitle: {result['title']}")
    print(f"Count: {result['count']}")
    print("Sample documents:")
    for sample in result["samples"]:
        print(f"  URL: {sample['url']}")
        print(f"  Date: {sample['date']}")
        print()

In [ ]:
# find articles with this title "Title: MIT Technology Review" (case insensitive)

collection.create_index([("title", "text")])

results = collection.find(
    {"$text": {"$search": "MIT Technology Review"}},
    {"_id": 0, "title": 1, "url": 1, "date": 1, "body": 1},
)

for result in results:
    print(f"Title: {result['title']}")
    print(f"URL: {result['url']}")
    print(f"Date: {result['date']}")
    print(f"Body: {result['body'][:200]}...")
    print()

In [ ]:
from datetime import datetime

pipeline = [
    {
        "$project": {
            "title": 1,
            "url": 1,
            "date": {"$dateToString": {"format": "%Y-%m-%d", "date": "$date"}},
        }
    },
    {
        "$group": {
            "_id": {"title": "$title", "date": "$date"},
            "count": {"$sum": 1},
            "urls": {"$push": "$url"},
        }
    },
    {"$match": {"count": {"$gt": 1}}},
    {
        "$group": {
            "_id": None,
            "totalDuplicates": {"$sum": "$count"},
            "uniqueDuplicateSets": {"$sum": 1},
            "details": {
                "$push": {
                    "title": "$_id.title",
                    "date": "$_id.date",
                    "count": "$count",
                    "urls": "$urls",
                }
            },
        }
    },
    {
        "$project": {
            "_id": 0,
            "totalDuplicates": 1,
            "uniqueDuplicateSets": 1,
            "details": {
                "$slice": [
                    "$details",
                    10,
                ]  # Limite les détails aux 10 premiers ensembles de doublons
            },
        }
    },
]

results = collection.aggregate(pipeline)

for result in results:
    print(f"Nombre total d'articles en double : {result['totalDuplicates']}")
    print(f"Nombre d'ensembles uniques de doublons : {result['uniqueDuplicateSets']}")
    print("\nExemples de doublons (10 premiers ensembles) :")
    for detail in result["details"]:
        print(f"\nTitre : {detail['title']}")
        print(f"Date : {detail['date']}")
        print(f"Nombre de doublons : {detail['count']}")
        print("URLs :")
        for url in detail["urls"][:3]:  # Affiche jusqu'à 3 URLs par ensemble
            print(f"  - {url}")
        if len(detail["urls"]) > 3:
            print(f"  ... et {len(detail['urls']) - 3} de plus")

In [ ]:
import json
from datetime import datetime
from bson import json_util  # pour gérer la sérialisation des types MongoDB

pipeline = [
    {
        "$project": {
            "title": 1,
            "url": 1,
            "date": {"$dateToString": {"format": "%Y-%m-%d", "date": "$date"}},
        }
    },
    {
        "$group": {
            "_id": {"title": "$title", "date": "$date"},
            "urls": {"$push": "$url"},
        }
    },
    {"$match": {"$expr": {"$gt": [{"$size": "$urls"}, 1]}}},
    {"$project": {"_id": 0, "title": "$_id.title", "date": "$_id.date", "urls": 1}},
    {"$sort": {"title": 1, "date": 1}},
]

results = list(collection.aggregate(pipeline))


# Fonction pour encoder les résultats en JSON
def json_encoder(obj):
    if isinstance(obj, datetime):
        return obj.isoformat()
    return json_util.default(obj)


# Sauvegarder les résultats dans un fichier JSON
with open("duplicates.json", "w", encoding="utf-8") as f:
    json.dump(results, f, ensure_ascii=False, indent=2, default=json_encoder)

print(f"Les données ont été sauvegardées dans 'duplicates.json'.")
print(f"Nombre total d'ensembles de doublons : {len(results)}")

# Afficher un exemple des premières entrées
print("\nExemple des 5 premières entrées :")
for entry in results[:5]:
    print(f"\nTitre : {entry['title']}")
    print(f"Date : {entry['date']}")
    print(f"Nombre d'URLs : {len(entry['urls'])}")
    print("Quelques URLs :")
    for url in entry["urls"][:3]:
        print(f"  - {url}")
    if len(entry["urls"]) > 3:
        print(f"  ... et {len(entry['urls']) - 3} de plus")